In [ ]:
import pandas as pd
import numpy as np

## Passes df

In [ ]:
# Filter for accurate passes
df2 = pd.read_csv("player_appearance_pass.csv")
df_received = df2[df2['accurate'] == True].copy()

# Drop rows where addressee_player_appearance_id is null and convert to integer
df_received = df_received.dropna(subset=['addressee_player_appearance_id'])
df_received['addressee_player_appearance_id'] = df_received['addressee_player_appearance_id'].astype(int)

df_received = df_received[df_received['stage'] == 'top']

# Function to assign minute blocks
def get_minute_block(minute):
    if minute <= 15:
        return 15
    elif 16 <= minute <= 30:
        return 30
    elif 31 <= minute:
        return 45

# Apply the minute blocks
df_received['minute_block'] = df_received['minute'].apply(get_minute_block)

ids = df_received['addressee_player_appearance_id'].unique()
periods = df_received['period'].unique()
minutes = df_received['minute_block'].unique()

# Create the Cartesian Product (all possible combinations)
full_index = pd.MultiIndex.from_product(
    [ids, periods, minutes],
    names=['addressee_player_appearance_id', 'period', 'minute_block']
)

# Group by the keys and sum the numeric columns
df_unique = df_received.groupby(['addressee_player_appearance_id', 'period', 'minute_block']).sum().reset_index()
df_unique.head()

df_expanded = df_unique.set_index(['addressee_player_appearance_id', 'period', 'minute_block']) \
                       .reindex(full_index) \
                       .reset_index()
df_expanded = df_expanded.iloc[:,0:3]

df_expanded = df_expanded.merge(df_received, how="left", on=['addressee_player_appearance_id', 'period', 'minute_block'])

# Group by the specified columns and count the passes
grouped = df_received.groupby(
    ['addressee_player_appearance_id', 'period', 'minute_block']
).size().reset_index(name='last15_received_passes')

# Map periods and minute blocks to numeric values ON THE GROUPED DATAFRAME ---
period_order = {'half_1': 1, 'half_2': 2, 'extra_time_1': 3}
minute_order = {15: 1, 30: 2, 45: 3}

grouped['period_sort'] = grouped['period'].map(period_order)
grouped['minute_sort'] = grouped['minute_block'].map(minute_order)

# Sort to calculate the cumulative sum in chronological order
grouped = grouped.sort_values(
    by=['addressee_player_appearance_id', 'period_sort', 'minute_sort']
).drop(columns=['period_sort', 'minute_sort'])

# Add the cumulative sum column per player
grouped['cumul_received_passes'] = grouped.groupby('addressee_player_appearance_id')['last15_received_passes'].cumsum()

df_pass = grouped

## Under pressure df

In [ ]:
# Load the dataset
df = pd.read_csv("player_appearance_behaviour_under_pressure.csv")

# Filter for the relevant periods
periods = ['half_1', 'half_2', 'extra_time_1']
df = df[df['period'].isin(periods)].copy()

# Function to assign minute blocks
def get_minute_block(minute):
    if minute <= 15:
        return 15
    elif 16 <= minute <= 30:
        return 30
    elif 31 <= minute:
        return 45

df['minute_block'] = df['minute'].apply(get_minute_block)

ids = df['player_appearance_id'].unique()
periods = df['period'].unique()
minutes = df['minute_block'].unique()

# Create the Cartesian Product (all possible combinations)
full_index = pd.MultiIndex.from_product(
    [ids, periods, minutes],
    names=['player_appearance_id', 'period', 'minute_block']
)

# Group by the keys and sum the numeric columns
df_unique = df.groupby(['player_appearance_id', 'period', 'minute_block']).sum().reset_index()
df_unique.head()
# Now the reindexing will work perfectly
df_expanded = df_unique.set_index(['player_appearance_id', 'period', 'minute_block']) \
                       .reindex(full_index) \
                       .reset_index()
df_expanded = df_expanded.iloc[:,0:3]

df_expanded = df_expanded.merge(df, how="left", on=['player_appearance_id', 'period', 'minute_block'])

# Aggregate data by player_appearance_id, period, and minute_block
agg_funcs = {
    'accurate': ['count', 'sum'],  # total actions, accurate actions
    'pass_angle': ['count', 'sum'] # valid angles count, sum of angles
}

grouped = df_expanded.groupby(['player_appearance_id', 'period', 'minute_block']).agg(agg_funcs).reset_index()

# Flatten MultiIndex columns
grouped.columns = ['player_appearance_id', 'period', 'minute_block',
                   'total_actions', 'accurate_actions',
                   'valid_angle_count', 'sum_angle']
grouped['accurate_actions'] = grouped['accurate_actions'].astype("int")

# Map periods and minute blocks to numeric values for correct chronological sorting
period_order = {'half_1': 1, 'half_2': 2, 'extra_time_1': 3}
minute_order = {15: 1, 30: 2, 45: 3}

grouped['period_sort'] = grouped['period'].map(period_order)
grouped['minute_sort'] = grouped['minute_block'].map(minute_order)

# Sort the grouped dataframe chronologically per player
grouped = grouped.sort_values(by=['player_appearance_id', 'period_sort', 'minute_sort']).drop(columns=['period_sort', 'minute_sort'])

# Calculate block metrics
grouped['last15_action_accuracy'] = np.where(grouped['total_actions']!= 0, grouped['accurate_actions'] / grouped['total_actions'], 0)
grouped['last15_average_angle'] = np.where(grouped['valid_angle_count'] > 0, grouped['sum_angle'] / grouped['valid_angle_count'], np.nan)

# Calculate cumulative moving sums per player
grouped['cum_total_actions'] = grouped.groupby('player_appearance_id')['total_actions'].cumsum()
grouped['cum_accurate_actions'] = grouped.groupby('player_appearance_id')['accurate_actions'].cumsum()
grouped['cum_valid_angle_count'] = grouped.groupby('player_appearance_id')['valid_angle_count'].cumsum()
grouped['cum_sum_angle'] = grouped.groupby('player_appearance_id')['sum_angle'].cumsum()

# Calculate cumulative metrics per player
grouped['cumul_action_accuracy'] = grouped['cum_accurate_actions'] / grouped['cum_total_actions']
grouped['cumul_average_angle'] = np.where(grouped['cum_valid_angle_count'] > 0, grouped['cum_sum_angle'] / grouped['cum_valid_angle_count'], np.nan)

# Select and rename final columns for clean output
final_cols = [
    'player_appearance_id', 'period', 'minute_block',
    'last15_action_accuracy', 'cumul_action_accuracy',
    'last15_average_angle', 'cumul_average_angle'
]
final_df = grouped[final_cols]

## Run

In [ ]:
# Load the dataset
df = pd.read_csv("player_appearance_run.csv")

# Filter for the relevant periods
periods = ['half_1', 'half_2', 'extra_time_1']
df = df[df['period'].isin(periods)].copy()

df = df[df['run_type'] == 'sprint']

# Function to assign minute blocks
def get_minute_block(minute):
    if minute <= 15:
        return 15
    elif 16 <= minute <= 30:
        return 30
    elif 31 <= minute:
        return 45

df['minute_block'] = df['minute'].apply(get_minute_block)

ids = df['player_appearance_id'].unique()
periods = df['period'].unique()
minutes = df['minute_block'].unique()

# Create the Cartesian Product (all possible combinations)
full_index = pd.MultiIndex.from_product(
    [ids, periods, minutes],
    names=['player_appearance_id', 'period', 'minute_block']
)

# Group by the keys and sum the numeric columns
df_unique = df.groupby(['player_appearance_id', 'period', 'minute_block']).sum().reset_index()
df_unique.head()

df_expanded = df_unique.set_index(['player_appearance_id', 'period', 'minute_block']) \
                       .reindex(full_index) \
                       .reset_index()
df_expanded = df_expanded.iloc[:,0:3]

In [ ]:
agg_funcs = {
    'distance': 'sum',
}

df_run = df_expanded.merge(df, how="left", on=['player_appearance_id', 'period', 'minute_block'])
grouped = df_run.groupby(['player_appearance_id', 'period', 'minute_block']).agg(agg_funcs).reset_index()

period_order = {'half_1': 1, 'half_2': 2, 'extra_time_1': 3}
minute_order = {15: 1, 30: 2, 45: 3}

grouped['period_sort'] = grouped['period'].map(period_order)
grouped['minute_sort'] = grouped['minute_block'].map(minute_order)

grouped = grouped.sort_values(
    by=['player_appearance_id', 'period_sort', 'minute_sort']
).drop(columns=['period_sort', 'minute_sort'])

grouped.columns = ['player_appearance_id', 'period', 'minute_block',
                   'last15_sprint_dist']

# Add the cumulative sum column per player
grouped['cumul_sprint_dist'] = grouped.groupby('player_appearance_id')['last15_sprint_dist'].cumsum()

df_run = grouped
df_run.head()

,player_appearance_id,period,minute_block,last15_sprint_dist,cumul_sprint_dist
3,39421,half_1,15,0.00000,0.00000
4,39421,half_1,30,0.00000,0.00000
5,39421,half_1,45,12.93474,12.93474
6,39421,half_2,15,0.00000,12.93474
7,39421,half_2,30,0.00000,12.93474


## Merge

In [ ]:
df = pd.read_csv("players_quarters_final.csv")

In [ ]:
df_merge1 = df.merge(df_pass, how="left", left_on=["player_appearance_id", "checkpoint_period", "checkpoint_min"],
                     right_on = ["addressee_player_appearance_id", "period", "minute_block"])
df_merge1.drop(columns = ["addressee_player_appearance_id", "period", "minute_block"], inplace=True)
df_merge1.head()

,player_appearance_id,player_id,fixture_id,date,checkpoint,checkpoint_period,checkpoint_min,position,is_home,formation,...,cumul_distance,cumul_mean_max_speed,cumul_peak_speed,cumul_shots,cumul_shots_on_target,cumul_shots_under_press,cumul_shots_top_third,scored_after,last15_received_passes,cumul_received_passes
0,39421,2702,1159,2025-06-11,H1_15,half_1,15,G,True,4-1-4-1,...,0.00,0.000,0.00,0,0,0,0,0,NaN,NaN
1,39422,2732,1159,2025-06-11,H1_15,half_1,15,D,True,4-1-4-1,...,68.15,6.806,7.46,0,0,0,0,0,NaN,NaN
2,39423,3296,1159,2025-06-11,H1_15,half_1,15,D,True,4-1-4-1,...,68.49,7.165,7.87,0,0,0,0,0,NaN,NaN
3,39424,3297,1159,2025-06-11,H1_15,half_1,15,D,True,4-1-4-1,...,80.36,6.873,8.95,0,0,0,0,0,NaN,NaN
4,39425,3298,1159,2025-06-11,H1_15,half_1,15,A,True,4-1-4-1,...,183.30,6.533,7.68,0,0,0,0,0,NaN,NaN


In [ ]:
df_merge2 = df_merge1.merge(final_df, how="left", left_on=["player_appearance_id", "checkpoint_period", "checkpoint_min"],
                            right_on = ["player_appearance_id", "period", "minute_block"])
df_merge2.drop(columns = ["period", "minute_block"], inplace=True)
df_merge2.head()

,player_appearance_id,player_id,fixture_id,date,checkpoint,checkpoint_period,checkpoint_min,position,is_home,formation,...,cumul_shots_on_target,cumul_shots_under_press,cumul_shots_top_third,scored_after,last15_received_passes,cumul_received_passes,last15_action_accuracy,cumul_action_accuracy,last15_average_angle,cumul_average_angle
0,39421,2702,1159,2025-06-11,H1_15,half_1,15,G,True,4-1-4-1,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,39422,2732,1159,2025-06-11,H1_15,half_1,15,D,True,4-1-4-1,...,0,0,0,0,NaN,NaN,0.50,0.50,16.115202,16.115202
2,39423,3296,1159,2025-06-11,H1_15,half_1,15,D,True,4-1-4-1,...,0,0,0,0,NaN,NaN,0.00,NaN,NaN,NaN
3,39424,3297,1159,2025-06-11,H1_15,half_1,15,D,True,4-1-4-1,...,0,0,0,0,NaN,NaN,1.00,1.00,-23.155529,-23.155529
4,39425,3298,1159,2025-06-11,H1_15,half_1,15,A,True,4-1-4-1,...,0,0,0,0,NaN,NaN,0.25,0.25,-36.154095,-36.154095


In [ ]:
df_merge3 = df_merge2.merge(df_run, how="left", left_on=["player_appearance_id", "checkpoint_period", "checkpoint_min"],
                            right_on = ["player_appearance_id", "period", "minute_block"])
df_merge3.drop(columns = ["period", "minute_block"], inplace=True)
df_merge3 = df_merge3.fillna(0)

In [ ]:
df_merge3 = df_merge3.iloc[:, np.r_[7, 10, 14:df_merge3.shape[1]]]
df_merge3["minute_in"] = (df_merge3["minute_in"]>1).astype(int)

In [ ]:
df_merge3['last15_sprint_ratio'] = df_merge3['last15_sprint_dist']/(df_merge3['last15_distance']+1)
df_merge3['last15_fatigue'] = df_merge3['last15_distance']/(df_merge3['cumul_distance']+1)

In [ ]:
score_col = df_merge3.pop('scored_after')
df_merge3['scored_after'] = score_col

In [ ]:
df_merge3.to_csv("very_final_dataset.csv", index=False)

In [ ]:
df_merge3.head()

,position,minute_in,last15_sprints,last15_hsr,last15_distance,last15_mean_max_speed,last15_peak_speed,last15_shots,last15_shots_on_target,last15_shots_under_press,...,cumul_received_passes,last15_action_accuracy,cumul_action_accuracy,last15_average_angle,cumul_average_angle,last15_sprint_dist,cumul_sprint_dist,last15_sprint_ratio,last15_fatigue,scored_after
0,G,0,0,0,0.00,0.000,0.00,0,0,0,...,0.0,0.00,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0
1,D,0,2,5,68.15,6.806,7.46,0,0,0,...,0.0,0.50,0.50,16.115202,16.115202,16.565102,16.565102,0.239553,0.985539,0
2,D,0,2,4,68.49,7.165,7.87,0,0,0,...,0.0,0.00,0.00,0.000000,0.000000,28.842802,28.842802,0.415064,0.985609,0
3,D,0,1,5,80.36,6.873,8.95,0,0,0,...,0.0,1.00,1.00,-23.155529,-23.155529,25.375646,25.375646,0.311893,0.987709,0
4,A,0,4,15,183.30,6.533,7.68,0,0,0,...,0.0,0.25,0.25,-36.154095,-36.154095,45.981952,45.981952,0.249495,0.994574,0
